# SuperbCommand – Multi-Asset CTA Strategy
## Notebook 01H — 2026 Holdout Data Extension — Corrected Holdout Downloader

**Purpose:** refresh the existing frozen market-data universe through the latest available
2026 observation so Notebook 18 can evaluate the untouched holdout.

This notebook is **not a research/optimisation notebook**. It does not change the universe,
signal parameters, or strategy rules.

### Holdout protocol
- Historical research specification remains frozen through **31 December 2025**.
- Data from **1 January 2026 onward** are holdout/evaluation data.
- The original pre-2026 processed files are backed up before any full-history evaluation
  files are written.
- No 2026 result is inspected here for model selection.

In [1]:

# ============================================================
# 1.1 — IMPORTS & DEPENDENCIES
# ============================================================

from __future__ import annotations

import json
import os
import platform
import subprocess
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# yfinance is usually available in Colab, but install it if necessary.
try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "yfinance"])
    import yfinance as yf

print("Python   :", sys.version.split()[0])
print("pandas   :", pd.__version__)
print("NumPy    :", np.__version__)
print("yfinance :", yf.__version__)


Python   : 3.13.15
pandas   : 2.2.3
NumPy    : 2.1.3
yfinance : 0.2.66


In [2]:

# ============================================================
# 1.2 — MOUNT GOOGLE DRIVE & LOAD NOTEBOOK 00 CONFIG
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Google Colab not detected. Using local fallback paths.")

if IN_COLAB:
    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/Colab Notebooks/"
        "SuperbCommand/Multi-Asset CTA Strategy"
    )
else:
    PROJECT_ROOT = Path.cwd() / "Multi-Asset CTA Strategy"

CONFIG_PATH = PROJECT_ROOT / "config" / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Notebook 00 configuration was not found. "
        "Run Notebook 00 — Research Configuration & Project Initialisation first. "
        f"Expected file: {CONFIG_PATH}"
    )

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PATHS = {name: Path(path) for name, path in PROJECT_CONFIG["paths"].items()}

RESEARCH_END = pd.Timestamp(PROJECT_CONFIG["research_dates"]["research_end"])
HOLDOUT_START = pd.Timestamp(PROJECT_CONFIG["research_dates"]["holdout_start"])
DEFAULT_RESEARCH_START = pd.Timestamp(
    PROJECT_CONFIG["research_dates"]["default_research_start"]
)
UNLOCK_HOLDOUT = bool(PROJECT_CONFIG["research_dates"]["unlock_holdout"])

if UNLOCK_HOLDOUT:
    raise RuntimeError(
        "Notebook 01 must not run with the final holdout unlocked. "
        "Set UNLOCK_HOLDOUT=False in the research configuration."
    )

print("Project root :", PROJECT_ROOT)
print("Research     :", DEFAULT_RESEARCH_START.date(), "to", RESEARCH_END.date())
print("Holdout from :", HOLDOUT_START.date(), "(LOCKED)")


Mounted at /content/drive
Project root : /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy
Research     : 2005-01-01 to 2025-12-31
Holdout from : 2026-01-01 (LOCKED)



## Initial prototype universe

The first research universe should be broad enough to test whether SuperbCommand generalises across fundamentally different markets, but small enough to remain interpretable and fast.

The initial **core** universe deliberately uses liquid, widely followed proxies.

### Equities
- SPY — S&P 500
- EFA — developed markets ex-US
- EEM — emerging markets

### Government rates
- SHY — 1–3 year US Treasuries
- IEF — 7–10 year US Treasuries
- TLT — 20+ year US Treasuries
- TIP — US inflation-linked Treasuries

### Commodities
- GLD — gold
- DBC — diversified commodity basket

### FX
- UUP — US Dollar Index proxy

### Crypto
- BTC-USD — Bitcoin spot price proxy

This creates an intentionally compact cross-asset laboratory. Later notebooks can expand the universe only after the basic research engine is functioning correctly.

A key limitation is that ETF histories begin at different dates. Notebook 01 therefore reports each instrument's usable history rather than silently forcing all markets to share one start date.


In [3]:

# ============================================================
# 1.3 — DEFINE PROTOTYPE UNIVERSE
# ============================================================

UNIVERSE = pd.DataFrame(
    [
        # Equities
        ("SPY",     "equities",   "US Large-Cap Equities",          "ETF",    "S&P 500 proxy"),
        ("EFA",     "equities",   "Developed ex-US Equities",       "ETF",    "MSCI EAFE proxy"),
        ("EEM",     "equities",   "Emerging-Market Equities",       "ETF",    "MSCI EM proxy"),

        # Rates
        ("SHY",     "rates",      "US Treasury 1-3Y",               "ETF",    "Short-duration Treasury proxy"),
        ("IEF",     "rates",      "US Treasury 7-10Y",              "ETF",    "Intermediate-duration Treasury proxy"),
        ("TLT",     "rates",      "US Treasury 20Y+",               "ETF",    "Long-duration Treasury proxy"),
        ("TIP",     "rates",      "US Inflation-Linked Treasuries", "ETF",    "TIPS proxy"),

        # Commodities
        ("GLD",     "commodities","Gold",                            "ETF",    "Physical gold proxy"),
        ("DBC",     "commodities","Broad Commodities",               "ETF",    "Diversified commodity-futures proxy"),

        # FX
        ("UUP",     "fx",         "US Dollar Index",                 "ETF",    "USD basket proxy"),

        # Crypto
        ("BTC-USD", "crypto",     "Bitcoin",                         "Spot",   "Yahoo Finance BTC/USD series"),
    ],
    columns=["ticker", "asset_class", "market_name", "instrument_type", "notes"],
).set_index("ticker")

display(UNIVERSE)

print("\nUniverse size:", len(UNIVERSE))
print("\nMarkets by asset class:")
print(UNIVERSE.groupby("asset_class").size().to_string())


,asset_class,market_name,instrument_type,notes
ticker,,,,
SPY,equities,US Large-Cap Equities,ETF,S&P 500 proxy
EFA,equities,Developed ex-US Equities,ETF,MSCI EAFE proxy
EEM,equities,Emerging-Market Equities,ETF,MSCI EM proxy
SHY,rates,US Treasury 1-3Y,ETF,Short-duration Treasury proxy
IEF,rates,US Treasury 7-10Y,ETF,Intermediate-duration Treasury proxy
TLT,rates,US Treasury 20Y+,ETF,Long-duration Treasury proxy
TIP,rates,US Inflation-Linked Treasuries,ETF,TIPS proxy
GLD,commodities,Gold,ETF,Physical gold proxy
DBC,commodities,Broad Commodities,ETF,Diversified commodity-futures proxy



Universe size: 11

Markets by asset class:
asset_class
commodities    2
crypto         1
equities       3
fx             1
rates          4



## Data acquisition policy

Yahoo Finance is being used here because it is fast and convenient for iterative research.

For Notebook 01:

- downloads stop at **31 December 2025**;
- post-2025 prices are never requested;
- raw downloads are cached to Google Drive;
- cleaned adjusted-close prices are stored separately;
- no missing observations are forward-filled across long gaps;
- the union of trading dates is preserved so non-US and crypto calendars are not artificially compressed;
- later strategy notebooks can decide whether to use daily, weekly or other frequencies.

For Yahoo Finance, the `end=` argument is exclusive, so the downloader requests **1 January 2026** while retaining only observations dated no later than **31 December 2025**.


In [4]:
# ============================================================
# 1H.4 — HOLDOUT DOWNLOAD SETTINGS
# ============================================================

DOWNLOAD_START = DEFAULT_RESEARCH_START
RESEARCH_END_FROZEN = RESEARCH_END
HOLDOUT_EVAL_END_EXCLUSIVE = pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
DOWNLOAD_END_EXCLUSIVE = HOLDOUT_EVAL_END_EXCLUSIVE

FORCE_REFRESH = True

RAW_PRICE_PATH = PATHS["data_raw"] / "prototype_yahoo_adjusted_close.parquet"
CLEAN_PRICE_PATH = PATHS["data_processed"] / "prototype_prices_daily.parquet"
RETURN_PATH = PATHS["data_processed"] / "prototype_returns_daily.parquet"
METADATA_PATH = PATHS["data_processed"] / "prototype_universe_metadata.csv"
QUALITY_PATH = PATHS["results"] / "prototype_data_quality.csv"

# Permanent pre-holdout backups, created only if absent.
BACKUP_DIR = PATHS["data_processed"] / "pre_2026_lockbox_backup"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print("Download start          :", DOWNLOAD_START.date())
print("Download end exclusive  :", DOWNLOAD_END_EXCLUSIVE.date())
print("Frozen research cutoff  :", RESEARCH_END_FROZEN.date())
print("Holdout start           :", HOLDOUT_START.date())
print("Force refresh           :", FORCE_REFRESH)

Download start          : 2005-01-01
Download end exclusive  : 2026-09-06
Frozen research cutoff  : 2025-12-31
Holdout start           : 2026-01-01
Force refresh           : True


In [5]:

# ============================================================
# 1.5 — ROBUST YAHOO DOWNLOAD FUNCTION
# ============================================================

def download_adjusted_close(
    tickers: Iterable[str],
    start: pd.Timestamp,
    end_exclusive: pd.Timestamp,
) -> pd.DataFrame:
    """Download adjusted-close-equivalent series from Yahoo Finance."""

    tickers = list(tickers)

    data = yf.download(
        tickers=tickers,
        start=str(pd.Timestamp(start).date()),
        end=str(pd.Timestamp(end_exclusive).date()),
        auto_adjust=False,
        actions=False,
        progress=False,
        threads=True,
        group_by="column",
    )

    if data is None or len(data) == 0:
        raise RuntimeError("Yahoo Finance returned no data.")

    # yfinance multi-ticker output normally has a MultiIndex:
    # first level = field, second level = ticker.
    if isinstance(data.columns, pd.MultiIndex):
        level0 = set(data.columns.get_level_values(0))

        if "Adj Close" in level0:
            prices = data["Adj Close"].copy()
        elif "Close" in level0:
            prices = data["Close"].copy()
        else:
            raise KeyError(
                f"Neither 'Adj Close' nor 'Close' found in Yahoo output. "
                f"Available fields: {sorted(level0)}"
            )
    else:
        # Defensive single-ticker fallback.
        field = "Adj Close" if "Adj Close" in data.columns else "Close"
        prices = data[[field]].copy()
        prices.columns = [tickers[0]]

    prices.index = pd.to_datetime(prices.index)
    prices.index.name = "date"

    # Standardise order and preserve unavailable tickers as NaN columns.
    prices = prices.reindex(columns=tickers)

    # ------------------------------------------------------------
    # 01H HOLDOUT EXTENSION:
    # Do NOT truncate at RESEARCH_END here.
    #
    # RESEARCH_END remains the frozen research/model-development
    # boundary, while this notebook is explicitly permitted to append
    # post-boundary observations for final holdout evaluation.
    # The yfinance `end` argument is exclusive, so additionally clip
    # defensively to dates strictly before end_exclusive.
    # ------------------------------------------------------------
    end_exclusive = pd.Timestamp(end_exclusive).normalize()
    prices = prices.loc[prices.index < end_exclusive]

    if len(prices) == 0:
        raise RuntimeError(
            "Yahoo download returned no usable observations through the "
            "requested holdout-evaluation end date."
        )

    # We require the extension actually to reach the holdout period.
    # This is the opposite of the locked research notebook's
    # contamination test: 01H is specifically designed to access 2026.
    if prices.index.max() < HOLDOUT_START:
        raise RuntimeError(
            "Downloaded data still do not reach HOLDOUT_START. "
            f"Latest observation: {prices.index.max().date()}, "
            f"holdout starts: {HOLDOUT_START.date()}."
        )

    return prices.sort_index()


def save_parquet_safe(df: pd.DataFrame, path: Path) -> None:
    """Save a DataFrame to parquet, installing pyarrow only if required."""
    try:
        df.to_parquet(path)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])
        df.to_parquet(path)


if RAW_PRICE_PATH.exists() and not FORCE_REFRESH:
    raw_prices = pd.read_parquet(RAW_PRICE_PATH)
    raw_prices.index = pd.to_datetime(raw_prices.index)
    print("Loaded cached Yahoo data:")
    print(RAW_PRICE_PATH)
else:
    raw_prices = download_adjusted_close(
        UNIVERSE.index,
        DOWNLOAD_START,
        DOWNLOAD_END_EXCLUSIVE,
    )
    save_parquet_safe(raw_prices, RAW_PRICE_PATH)
    print("Downloaded and cached Yahoo data:")
    print(RAW_PRICE_PATH)

print("\nRaw matrix shape:", raw_prices.shape)
print("First date:", raw_prices.index.min().date())
print("Last date :", raw_prices.index.max().date())


Downloaded and cached Yahoo data:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/raw/prototype_yahoo_adjusted_close.parquet

Raw matrix shape: (6815, 11)
First date: 2005-01-03
Last date : 2026-09-05



## Cleaning rules

The cleaned daily price matrix follows conservative rules:

1. remove duplicate timestamps;
2. sort chronologically;
3. coerce prices to numeric;
4. convert non-positive prices to missing;
5. remove rows where every market is missing;
6. preserve genuine gaps rather than filling them;
7. enforce the 2025 research cutoff again;
8. remove markets only if they contain no usable observations at all.

This is intentionally minimal. Signal-specific handling of missing observations belongs in later notebooks.


In [6]:
# ============================================================
# 1H.6 — CLEAN & STANDARDISE FULL EVALUATION PRICE MATRIX
# ============================================================

# Preserve original locked research files before replacing them with full-history
# evaluation files.
import shutil

for p in [RAW_PRICE_PATH, CLEAN_PRICE_PATH, RETURN_PATH, METADATA_PATH, QUALITY_PATH]:
    if p.exists():
        dst = BACKUP_DIR / p.name
        if not dst.exists():
            shutil.copy2(p, dst)
            print("Locked backup:", dst)

prices = raw_prices.copy()
prices = prices[~prices.index.duplicated(keep="last")]
prices = prices.sort_index()
prices = prices.apply(pd.to_numeric, errors="coerce")
prices = prices.mask(prices <= 0)
prices = prices.dropna(how="all")

# Evaluation data are allowed through the current download horizon.
prices = prices.loc[prices.index < HOLDOUT_EVAL_END_EXCLUSIVE]

all_missing = prices.columns[prices.notna().sum() == 0].tolist()
if all_missing:
    print("WARNING — no usable observations for:", all_missing)
    prices = prices.drop(columns=all_missing)

if prices.index.max() < HOLDOUT_START:
    raise RuntimeError("Refresh succeeded but still contains no 2026 holdout observations.")

save_parquet_safe(prices, CLEAN_PRICE_PATH)

print("Full evaluation price matrix:", prices.shape)
print("Latest observation:", prices.index.max().date())
print("Saved:", CLEAN_PRICE_PATH)

Full evaluation price matrix: (6815, 11)
Latest observation: 2026-09-05
Saved: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/processed/prototype_prices_daily.parquet



## Return construction

Daily simple returns are stored for diagnostics and later portfolio research.

Returns are calculated **per instrument without filling missing prices**. This avoids manufacturing zero returns during missing-data gaps.

The strategy itself may later operate on weekly or other resampled data; Notebook 01 does not impose a trading frequency.


In [7]:

# ============================================================
# 1.7 — DAILY RETURNS
# ============================================================

returns = prices.pct_change(fill_method=None)

# Infinite values should never survive into the research dataset.
returns = returns.replace([np.inf, -np.inf], np.nan)

save_parquet_safe(returns, RETURN_PATH)

print("Return matrix:", returns.shape)
print("Saved:", RETURN_PATH)


Return matrix: (6815, 11)
Saved: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/processed/prototype_returns_daily.parquet



## Coverage and data-quality diagnostics

The most important question at this stage is whether each market has enough usable history for the later walk-forward design.

For each instrument, Notebook 01 reports:

- first valid observation;
- last valid observation;
- number of valid observations;
- approximate years of history;
- missing observations within its active span;
- longest consecutive missing run;
- extreme one-day returns;
- latest usable price;
- whether the series reaches the 2025 research cutoff.

These diagnostics do not automatically reject shorter-history assets such as Bitcoin. They make the limitation explicit.


In [8]:

# ============================================================
# 1.8 — DATA-QUALITY REPORT
# ============================================================

def longest_missing_run(series: pd.Series) -> int:
    missing = series.isna().astype(int)
    if missing.sum() == 0:
        return 0
    groups = (missing != missing.shift()).cumsum()
    runs = missing.groupby(groups).sum()
    return int(runs.max())


quality_rows = []

for ticker in prices.columns:
    s = prices[ticker]
    valid = s.dropna()

    if len(valid) == 0:
        continue

    first_valid = valid.index.min()
    last_valid = valid.index.max()

    active = s.loc[first_valid:last_valid]
    r = returns[ticker].loc[first_valid:last_valid].dropna()

    span_days = (last_valid - first_valid).days
    years_history = span_days / 365.25 if span_days > 0 else 0.0

    missing_active = int(active.isna().sum())
    missing_active_pct = float(active.isna().mean())

    max_abs_daily_return = float(r.abs().max()) if len(r) else np.nan

    quality_rows.append(
        {
            "ticker": ticker,
            "asset_class": UNIVERSE.loc[ticker, "asset_class"],
            "market_name": UNIVERSE.loc[ticker, "market_name"],
            "first_valid": first_valid.date().isoformat(),
            "last_valid": last_valid.date().isoformat(),
            "valid_observations": int(valid.shape[0]),
            "years_history": round(years_history, 2),
            "missing_within_active_span": missing_active,
            "missing_pct_active_span": round(missing_active_pct, 4),
            "longest_missing_run": longest_missing_run(active),
            "max_abs_daily_return": round(max_abs_daily_return, 4)
                if pd.notna(max_abs_daily_return) else np.nan,
            "latest_price": round(float(valid.iloc[-1]), 6),
            "reaches_2025_cutoff": bool(
                last_valid >= (RESEARCH_END - pd.Timedelta(days=7))
            ),
        }
    )

quality = pd.DataFrame(quality_rows).set_index("ticker")
quality.to_csv(QUALITY_PATH)

display(quality)

print("\nSaved data-quality report:")
print(QUALITY_PATH)


,asset_class,market_name,first_valid,last_valid,valid_observations,years_history,missing_within_active_span,missing_pct_active_span,longest_missing_run,max_abs_daily_return,latest_price,reaches_2025_cutoff
ticker,,,,,,,,,,,,
SPY,equities,US Large-Cap Equities,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.1452,770.190002,True
EFA,equities,Developed ex-US Equities,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.1589,108.349998,True
EEM,equities,Emerging-Market Equities,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.2277,68.699997,True
SHY,rates,US Treasury 1-3Y,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.0071,81.690002,True
IEF,rates,US Treasury 7-10Y,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.0343,92.250000,True
TLT,rates,US Treasury 20Y+,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.0752,82.209999,True
TIP,rates,US Inflation-Linked Treasuries,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.0445,106.970001,True
GLD,commodities,Gold,2005-01-03,2026-09-04,5453,21.67,1361,0.1997,3,0.1129,406.769989,True
DBC,commodities,Broad Commodities,2006-02-06,2026-09-04,5178,20.57,1361,0.2081,3,0.0794,31.900000,True



Saved data-quality report:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/results/prototype_data_quality.csv


In [9]:

# ============================================================
# 1.9 — COVERAGE SUMMARY BY ASSET CLASS
# ============================================================

coverage_summary = (
    quality.reset_index()
    .groupby("asset_class")
    .agg(
        markets=("ticker", "count"),
        median_years_history=("years_history", "median"),
        min_years_history=("years_history", "min"),
        max_years_history=("years_history", "max"),
        median_valid_observations=("valid_observations", "median"),
    )
    .sort_index()
)

display(coverage_summary)


,markets,median_years_history,min_years_history,max_years_history,median_valid_observations
asset_class,,,,,
commodities,2,21.12,20.57,21.67,5315.5
crypto,1,11.97,11.97,11.97,4372.0
equities,3,21.67,21.67,21.67,5453.0
fx,1,19.51,19.51,19.51,4911.0
rates,4,21.67,21.67,21.67,5453.0



## Cross-sectional overlap

Because the universe contains instruments launched at different times, there are two useful notions of history:

### Instrument-specific history
Each market enters research when its own valid history begins.

### Fully overlapping history
The earliest date at which **all** prototype markets have simultaneous observations.

The fully overlapping sample will be much shorter because Bitcoin is young. Later notebooks should generally **not throw away older equity/rates/commodity history merely to accommodate Bitcoin**.

Instead, walk-forward research should permit a changing eligible universe subject to a minimum-history rule.


In [10]:

# ============================================================
# 1.10 — OVERLAP DIAGNOSTICS
# ============================================================

first_valid_dates = {
    ticker: prices[ticker].first_valid_index()
    for ticker in prices.columns
}

last_valid_dates = {
    ticker: prices[ticker].last_valid_index()
    for ticker in prices.columns
}

common_start = max(first_valid_dates.values())
common_end = min(last_valid_dates.values())

complete_case = prices.dropna(how="any")

print("Latest individual start date :", common_start.date())
print("Earliest individual end date :", common_end.date())

if len(complete_case):
    print("First complete-case row       :", complete_case.index.min().date())
    print("Last complete-case row        :", complete_case.index.max().date())
    print("Complete-case observations    :", len(complete_case))
else:
    print("No dates contain valid observations for every market simultaneously.")

print(
    "\nResearch policy: later notebooks should use an eligibility rule "
    "rather than forcing the whole project to begin at the latest market launch."
)


Latest individual start date : 2014-09-17
Earliest individual end date : 2026-09-04
First complete-case row       : 2014-09-17
Last complete-case row        : 2026-09-04
Complete-case observations    : 3010

Research policy: later notebooks should use an eligibility rule rather than forcing the whole project to begin at the latest market launch.



## Eligibility recommendation for later notebooks

A market should become eligible for signal/portfolio use only after it has accumulated sufficient history for the longest required indicator and risk-estimation window.

Notebook 01 does not yet know the final SuperbCommand parameterisation, so it does not impose a permanent rule.

A sensible later formulation is:

\[
\text{Eligible}_{i,t}
=
\mathbb{1}
\left(
N_{i,t}
\geq
N_{\min}
\right)
\]

where \(N_{\min}\) is determined by the maximum signal, volatility and covariance lookbacks plus an explicit safety buffer.

This prevents young series from being treated as though they had full historical information.


In [11]:

# ============================================================
# 1.11 — SAVE ENRICHED UNIVERSE METADATA
# ============================================================

metadata = UNIVERSE.copy()

metadata["first_valid"] = [
    quality.loc[ticker, "first_valid"] if ticker in quality.index else None
    for ticker in metadata.index
]
metadata["last_valid"] = [
    quality.loc[ticker, "last_valid"] if ticker in quality.index else None
    for ticker in metadata.index
]
metadata["years_history"] = [
    quality.loc[ticker, "years_history"] if ticker in quality.index else np.nan
    for ticker in metadata.index
]
metadata["valid_observations"] = [
    quality.loc[ticker, "valid_observations"] if ticker in quality.index else 0
    for ticker in metadata.index
]

metadata.to_csv(METADATA_PATH)

display(metadata)

print("\nSaved universe metadata:")
print(METADATA_PATH)


,asset_class,market_name,instrument_type,notes,first_valid,last_valid,years_history,valid_observations
ticker,,,,,,,,
SPY,equities,US Large-Cap Equities,ETF,S&P 500 proxy,2005-01-03,2026-09-04,21.67,5453
EFA,equities,Developed ex-US Equities,ETF,MSCI EAFE proxy,2005-01-03,2026-09-04,21.67,5453
EEM,equities,Emerging-Market Equities,ETF,MSCI EM proxy,2005-01-03,2026-09-04,21.67,5453
SHY,rates,US Treasury 1-3Y,ETF,Short-duration Treasury proxy,2005-01-03,2026-09-04,21.67,5453
IEF,rates,US Treasury 7-10Y,ETF,Intermediate-duration Treasury proxy,2005-01-03,2026-09-04,21.67,5453
TLT,rates,US Treasury 20Y+,ETF,Long-duration Treasury proxy,2005-01-03,2026-09-04,21.67,5453
TIP,rates,US Inflation-Linked Treasuries,ETF,TIPS proxy,2005-01-03,2026-09-04,21.67,5453
GLD,commodities,Gold,ETF,Physical gold proxy,2005-01-03,2026-09-04,21.67,5453
DBC,commodities,Broad Commodities,ETF,Diversified commodity-futures proxy,2006-02-06,2026-09-04,20.57,5178



Saved universe metadata:
/content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/data/processed/prototype_universe_metadata.csv



## Basic integrity tests

The notebook now verifies that:

- the dataset contains no post-2025 observations;
- all retained markets have positive prices;
- there are no duplicate dates;
- prices are chronological;
- returns contain no infinities;
- each canonical asset class is represented;
- saved output files exist.

A passing result is required before proceeding to SuperbCommand signal development.


In [12]:
# ============================================================
# 1H.12 — HOLDOUT-EXTENSION VALIDATION SUITE
# ============================================================

checks = {}
checks["nonempty_price_matrix"] = (len(prices) > 0 and prices.shape[1] > 0)
checks["pre_2026_history_present"] = bool((prices.index <= RESEARCH_END_FROZEN).any())
checks["2026_holdout_present"] = bool((prices.index >= HOLDOUT_START).any())
checks["unique_dates"] = prices.index.is_unique
checks["chronological_dates"] = prices.index.is_monotonic_increasing
checks["prices_positive_or_missing"] = bool(((prices > 0) | prices.isna()).all().all())
checks["returns_have_no_infinities"] = not np.isinf(returns.to_numpy(dtype=float)).any()

represented_classes = set(UNIVERSE.loc[prices.columns, "asset_class"])
required_classes = {"equities", "rates", "commodities", "fx", "crypto"}
checks["all_asset_classes_represented"] = required_classes.issubset(represented_classes)

checks["raw_file_written"] = RAW_PRICE_PATH.exists()
checks["clean_file_written"] = CLEAN_PRICE_PATH.exists()
checks["return_file_written"] = RETURN_PATH.exists()
checks["metadata_written"] = METADATA_PATH.exists()
checks["quality_report_written"] = QUALITY_PATH.exists()

validation = pd.DataFrame({"check": list(checks.keys()), "passed": list(checks.values())})
print(validation.to_string(index=False))

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise RuntimeError(f"Notebook 01H validation failed: {failed}")

print("\n" + "=" * 78)
print("NOTEBOOK 01H STATUS: PASS")
print("=" * 78)
print("2026 HOLDOUT MARKET DATA: ACCESSED FOR EVALUATION")
print("Latest data:", prices.index.max().date())
print("No strategy/model selection has occurred.")

                        check  passed
        nonempty_price_matrix    True
     pre_2026_history_present    True
         2026_holdout_present    True
                 unique_dates    True
          chronological_dates    True
   prices_positive_or_missing    True
   returns_have_no_infinities    True
all_asset_classes_represented    True
             raw_file_written    True
           clean_file_written    True
          return_file_written    True
             metadata_written    True
       quality_report_written    True

NOTEBOOK 01H STATUS: PASS
2026 HOLDOUT MARKET DATA: ACCESSED FOR EVALUATION
Latest data: 2026-09-05
No strategy/model selection has occurred.


In [13]:
# ============================================================
# 1H.13 — WRITE HOLDOUT DATA MANIFEST
# ============================================================

MANIFEST = {
    "notebook_id": "01H",
    "notebook_name": "2026 Holdout Data Extension",
    "project_name": "SuperbCommand – Multi-Asset CTA Strategy",
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "data_source": "Yahoo Finance via yfinance",
    "research_end_frozen": str(RESEARCH_END_FROZEN.date()),
    "holdout_start": str(HOLDOUT_START.date()),
    "holdout_accessed": True,
    "latest_dataset_date": str(prices.index.max().date()),
    "markets_retained": int(prices.shape[1]),
    "daily_rows": int(prices.shape[0]),
    "pre_2026_backup_dir": str(BACKUP_DIR),
    "optimisation_performed": False,
    "status": "PASS",
}

manifest_path = PATHS["manifests"] / "01H_2026_holdout_data_extension_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2)

print("Saved manifest:", manifest_path)

Saved manifest: /content/drive/MyDrive/Colab Notebooks/SuperbCommand/Multi-Asset CTA Strategy/manifests/01H_2026_holdout_data_extension_manifest.json



---

# End of Notebook 01

## Outputs created

- `data/raw/prototype_yahoo_adjusted_close.parquet`
- `data/processed/prototype_prices_daily.parquet`
- `data/processed/prototype_returns_daily.parquet`
- `data/processed/prototype_universe_metadata.csv`
- `results/prototype_data_quality.csv`
- `manifests/01_universe_and_data_engine_manifest.json`

## What Notebook 01 deliberately does not do

- calculate SuperbCommand;
- create trading signals;
- backtest positions;
- optimise hyperparameters;
- estimate portfolio weights;
- use 2026 data.

## Next notebook

**Notebook 02 — SuperbCommand Signal Engine**

Notebook 02 should reproduce the definitive SuperbCommand indicator in Python and verify that its state transitions behave exactly as intended before any portfolio optimisation is attempted.
